# BloomFace · Clasificador de cuatro gestos con LibreFace

Proyecto final de Procesamiento Digital de Imágenes. La CNN ResNet-18 oficial de LibreFace permanece congelada. Se entrena una cabeza lineal nueva sobre sus características y se compara con reglas de unidades de acción facial (AU). Los gestos describen movimientos visibles.

**En Colab** abre este cuaderno, conecta un entorno de CPU, ejecuta la instalación y reinicia el entorno si Colab lo solicita. Monta Drive si las fotos autorizadas ya están allí, o selecciona una carpeta temporal local de Colab. Ejecuta las celdas en orden. En Windows usa Python 3.12 y cambia las rutas de la primera sección. Ajusta `BLOOMFACE_DATA_DIR`, `BLOOMFACE_ASSETS_DIR` y `BLOOMFACE_OUTPUT_DIR` en el entorno o modifica los valores de la celda. La carpeta de datos debe contener los JPG/PNG del Parcial 2 y sus identificadores de persona, sesión y gesto.

| ID | Gesto | Efecto en BloomFace |
|---|---|---|
| 0 | reposo | Apertura media y escala base |
| 1 | sonrisa | Apertura completa y escala base |
| 2 | boca_abierta | Conserva la apertura al confirmar y aumenta a una escala fija |
| 3 | ceno_fruncido | Cierre y escala base |

La pérdida de rostro y la ambigüedad se gestionan como estados de interacción fuera de las cuatro salidas. Mantén los archivos faciales y el directorio de exportación fuera de Git.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("google") and importlib.util.find_spec("google.colab"):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "onnx==1.23.2", "onnxruntime==1.32.0", "mediapipe==1.2.3",
        "opencv-contrib-python==5.1.0.95", "scikit-learn==1.9.1", "Pillow",
    ])
    # Si vas a leer las fotos desde Drive, ejecuta estas líneas:
    # from google.colab import drive
    # drive.mount("/content/drive")

## 1. Datos y separación por persona

Se leen solo imágenes con etiquetas verificables por nombre (`pNN_sNN_gesto_NNN`). Los nombres sin persona o sesión quedan fuera de las métricas. Se eliminan duplicados exactos dentro del mismo grupo y se detiene el entrenamiento si hay duplicados entre personas o etiquetas incompatibles. Las personas se asignan antes de extraer características para que fotogramas cercanos de una misma captura permanezcan en la misma partición. Con tres personas el piloto usa una para cada partición. Cada persona se midió en una sesión, por lo que el piloto no estima variación entre sesiones de una misma persona. Con menos de tres personas completas solo se ejecutan las comprobaciones de preprocesamiento y pesos.

In [2]:
import csv
import hashlib
import itertools
import json
import os
import re
import unicodedata
from collections import Counter
from pathlib import Path

DATA_DIR = Path(os.environ.get("BLOOMFACE_DATA_DIR", "/content/Expresiones"))
ASSETS_DIR = Path(os.environ.get("BLOOMFACE_ASSETS_DIR", "/content/libreface"))
OUTPUT_DIR = Path(os.environ.get("BLOOMFACE_OUTPUT_DIR", "/content/bloomface-export"))
CHALLENGE_CSV = DATA_DIR / "pruebas_interaccion.csv"
LABELS = ("reposo", "sonrisa", "boca_abierta", "ceno_fruncido")
ALIASES = {name.replace("_", ""): idx for idx, name in enumerate(LABELS)}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
NAME = re.compile(r"^(p\d+)_(s\d+)_(.+)_\d+$", re.I)


def digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def normalize_label(text):
    text = unicodedata.normalize("NFKD", text.casefold())
    return "".join(char for char in text if char.isascii() and char.isalpha())


def inventory(directory):
    known_hashes = {}
    records = []
    ignored = 0
    repeated = 0
    if not directory.is_dir():
        print("Falta la carpeta de datos autorizados. Continúan las comprobaciones de pesos.")
        return records
    for path in sorted(directory.rglob("*")):
        if path.suffix.lower() not in IMAGE_SUFFIXES or not path.is_file():
            continue
        match = NAME.fullmatch(path.stem)
        if not match or normalize_label(match[3]) not in ALIASES:
            ignored += 1
            continue
        person, visit = match[1].lower(), match[2].lower()
        label_id = ALIASES[normalize_label(match[3])]
        key = digest(path)
        if key in known_hashes:
            if known_hashes[key] != (person, label_id):
                raise ValueError("La misma imagen figura en dos personas o clases. Corrige el inventario.")
            repeated += 1
            continue
        known_hashes[key] = (person, label_id)
        records.append((path, person, visit, label_id, key))
    print(f"Imágenes etiquetadas únicas: {len(records)}. Sin identificador o etiqueta: {ignored}. Duplicadas: {repeated}.")
    return records


def show_counts(records, title):
    print(title)
    for label_id, label in enumerate(LABELS):
        group = [row for row in records if row[3] == label_id]
        print(f"  {label:14} {len(group):4} imágenes · {len({row[1] for row in group})} personas · "
              f"{len({(row[1], row[2]) for row in group})} sesiones")


records = inventory(DATA_DIR)
show_counts(records, "Inventario")
people = sorted({r[1] for r in records})
observed = {p: {r[3] for r in records if r[1] == p} for p in people}
complete = [p for p in people if observed[p] == set(range(4))]
if len(complete) < 3:
    groups = {}
    print("Bloqueo de entrenamiento y exportación: se necesitan al menos tres personas con las cuatro clases.")
else:
    reserved = max(1, len(complete) // 5)
    groups = {
        "train": set(people) - set(complete[-2 * reserved:]),
        "val": set(complete[-2 * reserved:-reserved]),
        "test": set(complete[-reserved:]),
    }
    assert not any(a & b for a, b in itertools.combinations(groups.values(), 2))
    for role, participants in groups.items():
        print(f"{role}: {len(participants)} persona(s)")
        show_counts([r for r in records if r[1] in participants], "  Distribución")

Imágenes etiquetadas únicas: 898. Sin identificador o etiqueta: 4. Duplicadas: 0.
Inventario
  reposo          225 imágenes · 3 personas · 3 sesiones
  sonrisa         223 imágenes · 3 personas · 3 sesiones
  boca_abierta    225 imágenes · 3 personas · 3 sesiones
  ceno_fruncido   225 imágenes · 3 personas · 3 sesiones
train: 1 persona(s)
  Distribución
  reposo           75 imágenes · 1 personas · 1 sesiones
  sonrisa          73 imágenes · 1 personas · 1 sesiones
  boca_abierta     75 imágenes · 1 personas · 1 sesiones
  ceno_fruncido    75 imágenes · 1 personas · 1 sesiones
val: 1 persona(s)
  Distribución
  reposo           75 imágenes · 1 personas · 1 sesiones
  sonrisa          75 imágenes · 1 personas · 1 sesiones
  boca_abierta     75 imágenes · 1 personas · 1 sesiones
  ceno_fruncido    75 imágenes · 1 personas · 1 sesiones
test: 1 persona(s)
  Distribución
  reposo           75 imágenes · 1 personas · 1 sesiones
  sonrisa          75 imágenes · 1 personas · 1 sesiones
  boca_

## 2. Detección, alineación y contrato del encoder

Se detecta un único rostro mediante MediaPipe Face Landmarker en modo imagen. Para detectar, se reduce el lado mayor a un máximo de 640 píxeles. Las coordenadas relativas regresan a la imagen RGB original. La geometría de alineación toma los centros de los ojos y los extremos de la boca, genera un cuadrado orientado de 256 píxeles y extrae el centro de 224 × 224. La implementación aplica la geometría de OpenSense con muestreo bilineal y borde reflectante de OpenCV. El mismo código se usa en la aplicación y el cuaderno.

El encoder recibe `float32` RGB NCHW. Se divide por 255 y se aplica media y desviación de ImageNet por canal. Se imprime el contrato real de los dos ONNX. La entrada de referencia para paridad se selecciona de la partición de prueba y queda identificada por su SHA-256. El cuaderno omite su imagen, sus landmarks y su ruta en las salidas.

[Artículo de LibreFace](https://openaccess.thecvf.com/content/WACV2024/html/Chang_LibreFace_An_Open-Source_Toolkit_for_Deep_Facial_Expression_Analysis_WACV_2024_paper.html) · [Pesos oficiales de OpenSense](https://github.com/ihp-lab/OpenSense/tree/master/Utilities/LibreFace) · [Normalización ImageInput.cs](https://github.com/ihp-lab/OpenSense/blob/master/Utilities/LibreFace/ImageInput.cs) · [Geometría del alineador](https://github.com/ihp-lab/OpenSense/blob/master/Components/LibreFace/FaceImageAligner.cs) · [API de MediaPipe](https://developers.google.com/edge/mediapipe/solutions/vision/face_landmarker/python)

In [3]:
from pathlib import Path
import hashlib
import urllib.request

import cv2
import mediapipe as mp
import numpy as np
import onnxruntime as ort
from PIL import Image, ImageOps

CLASSES = ("reposo", "sonrisa", "boca_abierta", "ceno_fruncido")
AU_ORDER = (1, 2, 4, 5, 6, 9, 12, 15, 17, 20, 25, 26)
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
OPENSENSE = "https://media.githubusercontent.com/media/ihp-lab/OpenSense/master/Utilities/LibreFace/"
ARTIFACTS = {
    "LibreFace_AU_Encoder.onnx": (OPENSENSE + "LibreFace_AU_Encoder.onnx", "2eaf01eb6dffe3b6f50f8e84eb1ecf315e332c972fd77e12187a7bb956575d8c"),
    "LibreFace_AU_Intensity.onnx": (OPENSENSE + "LibreFace_AU_Intensity.onnx", "9c0c23a826e326ff6a5c9545e9792ec1624d0254588e5f3ab3bfef0e270dc374"),
    "face_landmarker.task": ("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task", "64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff"),
}
LEFT_EYE = (33, 7, 163, 144, 145, 153, 154, 155, 133, 246, 161, 160, 159, 158, 157, 173)
RIGHT_EYE = (263, 249, 390, 373, 374, 380, 381, 382, 362, 466, 388, 387, 386, 385, 384, 398)
LIPS = (61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 409, 270, 269, 267, 0, 37, 39, 40, 185)
PREPROCESS = {
    "version": 1,
    "detector_max_side": 640,
    "detector_max_faces": 2,
    "alignment_size": 256,
    "input_size": 224,
    "channels": "RGB",
    "layout": "NCHW",
    "mean": MEAN.tolist(),
    "std": STD.tolist(),
    "mouth_landmarks": list(LIPS),
    "eye_landmarks": [list(LEFT_EYE), list(RIGHT_EYE)],
}


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def verify_assets(directory, *, download=False):
    directory = Path(directory)
    if download:
        directory.mkdir(parents=True, exist_ok=True)
    for name, (url, expected) in ARTIFACTS.items():
        path = directory / name
        if not path.is_file():
            if not download:
                raise FileNotFoundError(f"Falta el archivo {path}")
            with urllib.request.urlopen(url, timeout=60) as response:
                content = response.read()
            if hashlib.sha256(content).hexdigest() != expected:
                raise ValueError(f"Descarga con un SHA-256 inesperado: {name}")
            path.write_bytes(content)
        if sha256(path) != expected:
            raise ValueError(f"SHA-256 inesperado: {path}")


def session(path, threads=1):
    options = ort.SessionOptions()
    options.intra_op_num_threads = threads
    options.inter_op_num_threads = 1
    return ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])


def read_rgb(path):
    with Image.open(path) as image:
        return np.asarray(ImageOps.exif_transpose(image).convert("RGB"))


def align_face(rgb, landmarks):
    height, width = rgb.shape[:2]
    points = np.array([(pt.x * width, pt.y * height) for pt in landmarks])
    left = points[list(LEFT_EYE)].mean(axis=0)
    right = points[list(RIGHT_EYE)].mean(axis=0)
    eye_center = (left + right) / 2
    lips = points[list(LIPS)]
    mouth = (lips[lips[:, 0].argmin()] + lips[lips[:, 0].argmax()]) / 2
    eye_to_eye = right - left
    eye_to_mouth = mouth - eye_center
    horizontal = eye_to_eye - np.array([-eye_to_mouth[1], eye_to_mouth[0]])
    norm = np.linalg.norm(horizontal)
    if not np.isfinite(norm) or norm < 1:
        return None
    horizontal *= max(2 * np.linalg.norm(eye_to_eye), 1.8 * np.linalg.norm(eye_to_mouth)) / norm
    vertical = np.array([-horizontal[1], horizontal[0]])
    center = eye_center + 0.1 * eye_to_mouth
    source = np.float32([center - horizontal - vertical, center - horizontal + vertical, center + horizontal + vertical])
    size = PREPROCESS["alignment_size"]
    target = np.float32([[0, 0], [0, size - 1], [size - 1, size - 1]])
    matrix = cv2.getAffineTransform(source, target)
    aligned = cv2.warpAffine(rgb, matrix, (size, size), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    crop = PREPROCESS["input_size"]
    offset = (size - crop) // 2
    aligned = aligned[offset:offset + crop, offset:offset + crop]
    normalized = (aligned.astype(np.float32) / 255 - MEAN) / STD
    return np.ascontiguousarray(normalized.transpose(2, 0, 1)[None])


class LibreFace:
    def __init__(self, directory):
        directory = Path(directory)
        verify_assets(directory)
        self.encoder = session(directory / "LibreFace_AU_Encoder.onnx")
        self.intensity = session(directory / "LibreFace_AU_Intensity.onnx")
        if self.encoder.get_inputs()[0].name != "image" or self.encoder.get_inputs()[0].shape[-3:] != [3, 224, 224]:
            raise ValueError("La entrada del encoder no cumple el contrato RGB 224×224.")
        vision = mp.tasks.vision
        options = vision.FaceLandmarkerOptions(
            base_options=mp.tasks.BaseOptions(model_asset_path=str(directory / "face_landmarker.task")),
            running_mode=vision.RunningMode.IMAGE,
            num_faces=PREPROCESS["detector_max_faces"],
        )
        self.detector = vision.FaceLandmarker.create_from_options(options)

    def close(self):
        self.detector.close()

    def prepare(self, rgb):
        height, width = rgb.shape[:2]
        ratio = min(1.0, PREPROCESS["detector_max_side"] / max(height, width))
        small = cv2.resize(rgb, (max(1, round(width * ratio)), max(1, round(height * ratio))), interpolation=cv2.INTER_AREA) if ratio < 1 else rgb
        result = self.detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(small)))
        if len(result.face_landmarks) != 1:
            return None
        return align_face(rgb, result.face_landmarks[0])

    def extract(self, tensor):
        feature = self.encoder.run(["feature"], {"image": tensor})[0]
        vector = feature.reshape(1, -1)
        raw_au = self.intensity.run(["au_intensity"], {"feature": vector})[0].reshape(-1)
        if len(raw_au) != len(AU_ORDER):
            raise ValueError("La cabeza de acciones faciales debe entregar doce intensidades.")
        return vector[0], np.clip(5 * raw_au, 0, 5)


def gesture_strength(au):
    brow, smile, lips, jaw = np.take(au, [AU_ORDER.index(n) for n in (4, 12, 25, 26)], axis=-1).T
    return np.stack((smile, np.maximum(lips, jaw), brow), axis=-1)


def rule_ids(strength, thresholds):
    active = strength >= np.asarray(thresholds)
    count = active.sum(axis=-1)
    return np.where(count == 0, 0, np.where(count == 1, active.argmax(axis=-1) + 1, -1))


def probabilities(logits):
    exp = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    return exp / exp.sum(axis=-1, keepdims=True)


def classify_scores(scores, au, policy):
    scores = np.asarray(scores)
    best = scores.argmax(axis=-1)
    top = np.sort(scores, axis=-1)[..., -2:]
    reject = (top[..., 1] < policy["min_score"]) | (top[..., 1] - top[..., 0] < policy["min_margin"])
    reject |= (gesture_strength(au) >= np.array(policy["combination_au_min"])).sum(axis=-1) > 1
    return np.where(reject, -1, best)

In [4]:
import platform
import time

from sklearn.metrics import confusion_matrix, f1_score, recall_score
from sklearn.linear_model import LogisticRegression

assert CLASSES == LABELS
if "analyzer" in globals():
    analyzer.close()
verify_assets(ASSETS_DIR, download=True)
analyzer = LibreFace(ASSETS_DIR)
print(f"Python {platform.python_version()} · {platform.system()} · ORT {ort.__version__} · OpenCV {cv2.__version__} · MediaPipe {mp.__version__}")
for name, model in (("encoder", analyzer.encoder), ("intensidades", analyzer.intensity)):
    print(name)
    for node in model.get_inputs() + model.get_outputs():
        print(f"  {node.name}: {node.type} {node.shape}")

assert analyzer.prepare(np.zeros((480, 640, 3), dtype=np.uint8)) is None
print("Prueba técnica: fotograma vacío rechazado por el detector.")
usable = {}
if not groups and records:
    tensor = analyzer.prepare(read_rgb(records[0][0]))
    if tensor is not None:
        vector, intensities = analyzer.extract(tensor)
        print(f"Prueba técnica: tensor {tensor.shape} y {vector.size} características finitas.")
        assert np.isfinite(vector).all() and np.isfinite(intensities).all()
elif groups:
    missing_by_role = Counter()
    corrupt_by_role = Counter()
    elapsed = []
    for role, participants in groups.items():
        bundle = []
        for record in records:
            path, person, *_ = record
            if person not in participants:
                continue
            try:
                rgb = read_rgb(path)
            except (OSError, ValueError):
                corrupt_by_role[role] += 1
                continue
            started = time.perf_counter()
            tensor = analyzer.prepare(rgb)
            if tensor is None:
                missing_by_role[role] += 1
                continue
            feature, au = analyzer.extract(tensor)
            assert tensor.shape == (1, 3, 224, 224) and tensor.dtype == np.float32
            assert np.isfinite(feature).all() and np.isfinite(au).all()
            elapsed.append(1000 * (time.perf_counter() - started))
            bundle.append((record, feature, au))
        usable[role] = bundle
        print(f"{role}: {len(bundle)} válidas, {missing_by_role[role]} sin rostro único, {corrupt_by_role[role]} ilegibles.")
    if elapsed:
        print(f"Preparar + encoder + AU en CPU: mediana {np.median(elapsed):.1f} ms y P95 {np.percentile(elapsed, 95):.1f} ms por imagen válida, sin lectura ni webcam.")
    ready = all({entry[0][3] for entry in bundle} == set(range(4)) for bundle in usable.values())
    if not ready:
        print("Bloqueo: alguna partición perdió una clase tras verificar el rostro.")
else:
    ready = False
if not groups:
    ready = False

Python 3.12.10 · Windows · ORT 1.32.0 · OpenCV 5.1.0 · MediaPipe 1.2.3
encoder
  image: tensor(float) [1, 3, 224, 224]
  feature: tensor(float) [1, 512, 1, 1]
intensidades
  feature: tensor(float) [1, 512]
  au_intensity: tensor(float) [1, 12]
Prueba técnica: fotograma vacío rechazado por el detector.
train: 298 válidas, 0 sin rostro único, 0 ilegibles.
val: 300 válidas, 0 sin rostro único, 0 ilegibles.
test: 300 válidas, 0 sin rostro único, 0 ilegibles.
Preparar + encoder + AU en CPU: mediana 45.2 ms y P95 58.0 ms por imagen válida, sin lectura ni webcam.


## 3. Referencia mediante unidades de acción

La cabeza oficial entrega doce intensidades, escaladas de 0 a 5. La referencia compara AU12 (sonrisa), el máximo de AU25 y AU26 (boca abierta), y AU4 (ceño). Cero señales activas da reposo. Dos o más dan abstención. Los umbrales se eligen por F1 macro en validación. Una abstención cuenta como error de la clase real.

[Orden oficial de las salidas y escala](https://github.com/ihp-lab/OpenSense/blob/master/Utilities/LibreFace/ActionUnitIntensityOutput.cs)

In [5]:
def arrays(role):
    bundle = usable[role]
    features = np.stack([item[1] for item in bundle]).astype(np.float32)
    au = np.stack([item[2] for item in bundle])
    targets = np.array([item[0][3] for item in bundle])
    return features, au, targets


def macro_f1(y, predicted):
    return f1_score(y, predicted, labels=list(range(4)), average="macro", zero_division=0)


if ready:
    x_train, au_train, y_train = arrays("train")
    x_val, au_val, y_val = arrays("val")
    val_strength = gesture_strength(au_val)
    for number in (4, 12, 25, 26):
        values = au_val[:, AU_ORDER.index(number)]
        print(f"AU{number} en validación: mínimo {values.min():.2f}, mediana {np.median(values):.2f}, máximo {values.max():.2f}")
    candidates = list(itertools.product(np.arange(0.5, 4.51, 0.5), repeat=3))
    rule_thresholds = max(candidates, key=lambda levels: macro_f1(y_val, rule_ids(val_strength, levels)))
    print("Umbrales de referencia (AU12, máximo AU25/AU26, AU4):", [float(v) for v in rule_thresholds])
    print(f"F1 macro con reglas en validación: {macro_f1(y_val, rule_ids(val_strength, rule_thresholds)):.4f}")

AU4 en validación: mínimo 0.02, mediana 0.18, máximo 1.36
AU12 en validación: mínimo 0.00, mediana 0.04, máximo 0.92
AU25 en validación: mínimo 0.00, mediana 0.01, máximo 1.62
AU26 en validación: mínimo 0.02, mediana 0.06, máximo 3.36
Umbrales de referencia (AU12, máximo AU25/AU26, AU4): [0.5, 0.5, 0.5]
F1 macro con reglas en validación: 0.4735


## 4. Entrenamiento de la cabeza BloomFace

El encoder y su cabeza AU oficial son solo de inferencia. Se entrena una regresión logística multinomial con pesos balanceados y cuatro logits en el orden acordado. Se elige la regularización por F1 macro de validación. La política de interacción aplica un mínimo de puntaje, un margen entre las dos clases principales y una señal simultánea de al menos dos gestos según las AU. Se elige la política sobre imágenes puras de validación. La detección de combinaciones y habla requiere pruebas específicas.

In [6]:
if ready:
    candidates = [
        LogisticRegression(C=c, max_iter=2000, class_weight="balanced").fit(x_train, y_train)
        for c in (0.01, 0.1, 1.0, 10.0)
    ]
    head = max(candidates, key=lambda fitted: macro_f1(y_val, fitted.predict(x_val)))
    assert head.classes_.tolist() == list(range(4))
    val_scores = probabilities(head.decision_function(x_val))

    policies = [
        {"min_score": score, "min_margin": margin, "combination_au_min": [level] * 3}
        for score, margin, level in itertools.product((0.5, 0.65, 0.8), (0.12, 0.2, 0.35), (2.5, 3.0, 3.5, 4.0))
    ]
    policy = max(
        policies,
        key=lambda settings: (
            macro_f1(y_val, classify_scores(val_scores, au_val, settings)),
            settings["min_score"], settings["min_margin"],
            -settings["combination_au_min"][0],
        ),
    )
    print(f"Regularización C={head.C:g}. F1 macro en validación sin abstención: {macro_f1(y_val, head.predict(x_val)):.4f}.")
    print(f"Política seleccionada: {policy}. F1 macro en validación con abstención: {macro_f1(y_val, classify_scores(val_scores, au_val, policy)):.4f}.")

Regularización C=0.01. F1 macro en validación sin abstención: 0.4839.
Política seleccionada: {'min_score': 0.65, 'min_margin': 0.2, 'combination_au_min': [2.5, 2.5, 2.5]}. F1 macro en validación con abstención: 0.4675.


## 5. Evaluación e interacción

Las tres matrices usan las mismas imágenes de la prueba reservada, incluidas las imágenes sin rostro único como abstenciones. Las filas son las clases reales, en el orden de BloomFace. La última columna registra abstención (incluye fallos de preprocesamiento). El F1 macro y el recall por clase incluyen esos errores. La pequeña cantidad de personas y la correlación de fotogramas limitan la interpretación.

La prueba de pérdida de rostro es una comprobación técnica con un fotograma vacío. Para documentar habla y gestos combinados con fotos autorizadas nuevas, agrega un CSV opcional `pruebas_interaccion.csv` bajo la carpeta de datos con columnas `archivo,caso` y casos `habla`, `gesto_combinado` o `sin_rostro`. Se usa una ruta relativa al conjunto, y se excluyen copias exactas de imágenes etiquetadas. Evaluar la confirmación temporal mientras alguien habla exige vídeo o una prueba de webcam. Un fotograma aislado puede ser visualmente indistinguible de boca abierta.

In [7]:
if ready:
    x_test, au_test, y_test = arrays("test")
    test_scores = probabilities(head.decision_function(x_test))
    test_rows = [row for row in records if row[1] in groups["test"]]
    test_targets = np.array([row[3] for row in test_rows])
    test_index = {item[0][4]: idx for idx, item in enumerate(usable["test"])}
    print(f"Prueba: {len(groups['test'])} persona(s), {len(test_rows)} imágenes, {len(x_test)} con rostro único.")

    evaluations = {
        "Reglas AU": rule_ids(gesture_strength(au_test), rule_thresholds),
        "Cabeza, argmax": test_scores.argmax(axis=1),
        "Cabeza, con abstención": classify_scores(test_scores, au_test, policy),
    }
    for name, ids in evaluations.items():
        prediction = np.array([ids[test_index[row[4]]] if row[4] in test_index else -1 for row in test_rows])
        matrix = confusion_matrix(test_targets, prediction, labels=[0, 1, 2, 3, -1])[:4]
        recalls = recall_score(test_targets, prediction, labels=list(range(4)), average=None, zero_division=0)
        print(f"\n{name} · F1 macro {macro_f1(test_targets, prediction):.4f} · abstenciones {sum(prediction == -1)}")
        for label, row, recall in zip(CLASSES, matrix, recalls):
            print(f"  {label:14} {str(row.tolist()):24} recall {recall:.4f}")
        print(f"  Sonrisa → boca abierta: {matrix[1, 2]}. Boca abierta → sonrisa: {matrix[2, 1]}.")

    if CHALLENGE_CSV.is_file():
        challenges = {"habla", "gesto_combinado", "sin_rostro"}
        counts = Counter()
        known = {row[4] for row in records}
        with CHALLENGE_CSV.open(encoding="utf-8-sig", newline="") as stream:
            for entry in csv.DictReader(stream):
                path = (DATA_DIR / entry["archivo"]).resolve()
                case = entry["caso"]
                if case not in challenges or not path.is_relative_to(DATA_DIR.resolve()):
                    raise ValueError("El manifiesto opcional contiene un caso o una ruta inválidos.")
                key = digest(path)
                if key in known:
                    raise ValueError("El manifiesto opcional contiene una imagen usada en clasificación o repetida.")
                known.add(key)
                tensor = analyzer.prepare(read_rgb(path))
                if tensor is None:
                    outcome = "sin rostro válido"
                else:
                    features, intensities = analyzer.extract(tensor)
                    decision = classify_scores(probabilities(head.decision_function(features[None])), intensities, policy).item()
                    outcome = "abstención" if decision == -1 else CLASSES[decision]
                counts[case, outcome] += 1
        for (case, outcome), count in sorted(counts.items()):
            print(f"Interacción · {case}: {outcome} ({count})")
    else:
        print("Interacción: fotograma vacío verificado. Habla y gestos combinados pendientes de material anotado o prueba en vivo.")

Prueba: 1 persona(s), 300 imágenes, 300 con rostro único.

Reglas AU · F1 macro 0.7282 · abstenciones 34
  reposo         [75, 0, 0, 0, 0]         recall 1.0000
  sonrisa        [46, 29, 0, 0, 0]        recall 0.3867
  boca_abierta   [2, 0, 43, 0, 30]        recall 0.5733
  ceno_fruncido  [7, 2, 0, 62, 4]         recall 0.8267
  Sonrisa → boca abierta: 0. Boca abierta → sonrisa: 0.

Cabeza, argmax · F1 macro 0.7262 · abstenciones 0
  reposo         [72, 2, 0, 1, 0]         recall 0.9600
  sonrisa        [28, 47, 0, 0, 0]        recall 0.6267
  boca_abierta   [2, 3, 68, 2, 0]         recall 0.9067
  ceno_fruncido  [33, 10, 0, 32, 0]       recall 0.4267
  Sonrisa → boca abierta: 0. Boca abierta → sonrisa: 3.

Cabeza, con abstención · F1 macro 0.6498 · abstenciones 74
  reposo         [61, 1, 0, 0, 13]        recall 0.8133
  sonrisa        [19, 40, 0, 0, 16]       recall 0.5333
  boca_abierta   [0, 0, 65, 0, 10]        recall 0.8667
  ceno_fruncido  [22, 4, 0, 14, 35]       recall 0.1867


## 6. Exportación ONNX y prueba de paridad

El clasificador exportado recibe vectores de características del encoder y devuelve cuatro logits. Se verifica la equivalencia de puntajes, clases y decisiones de abstención en **todas** las imágenes procesables de la persona de prueba. Se agrega una referencia de paridad sin imagen, con identidad SHA-256, cuatro puntajes y decisión esperada. El cuaderno conserva las métricas agregadas. Las fotos, sus recortes, los landmarks y los embeddings faciales quedan en el entorno privado.

Se pueden compartir el código, el cuaderno con resultados agregados y los enlaces a pesos. Si se redistribuyen los modelos oficiales, revisa la licencia aplicable, preserva los avisos y conserva la atribución. Comparte la cabeza derivada según la autorización de uso de los datos del equipo. La exportación local no autoriza publicar pesos ni rostros. [Licencia de LibreFace 2.0.0](https://www.nuget.org/packages/LibreFace/2.0.0/License) · [Modelo MediaPipe Face Landmarker](https://developers.google.com/edge/mediapipe/solutions/vision/face_landmarker)

In [8]:
import onnx
from onnx import TensorProto, helper, numpy_helper

if ready:
    features = x_test.shape[1]
    graph = helper.make_graph(
        [helper.make_node("Gemm", ["feature", "weight", "bias"], ["logits"])],
        "BloomFaceLinearHead",
        [helper.make_tensor_value_info("feature", TensorProto.FLOAT, [None, features])],
        [helper.make_tensor_value_info("logits", TensorProto.FLOAT, [None, len(CLASSES)])],
        [
            numpy_helper.from_array(np.ascontiguousarray(head.coef_.T, dtype=np.float32), "weight"),
            numpy_helper.from_array(np.asarray(head.intercept_, dtype=np.float32), "bias"),
        ],
    )
    model = helper.make_model(graph, producer_name="BloomFace", ir_version=9, opset_imports=[helper.make_opsetid("", 17)])
    onnx.checker.check_model(model)
    exported = ort.InferenceSession(model.SerializeToString(), providers=["CPUExecutionProvider"])
    exported_scores = probabilities(exported.run(["logits"], {"feature": x_test})[0])
    exported_ids = classify_scores(exported_scores, au_test, policy)
    np.testing.assert_allclose(exported_scores, test_scores, rtol=1e-5, atol=1e-6)
    np.testing.assert_array_equal(exported_scores.argmax(axis=1), test_scores.argmax(axis=1))
    np.testing.assert_array_equal(exported_ids, classify_scores(test_scores, au_test, policy))
    reference_index = int(np.flatnonzero((exported_ids == y_test) & (y_test == 2))[0]) if np.any((exported_ids == y_test) & (y_test == 2)) else int(np.argmax(exported_scores.max(axis=1)))
    entry = usable["test"][reference_index][0]
    fresh = analyzer.prepare(read_rgb(entry[0]))
    ref_vector, ref_au = analyzer.extract(fresh)
    np.testing.assert_allclose(ref_vector, x_test[reference_index], rtol=1e-5, atol=1e-6)
    np.testing.assert_allclose(ref_au, au_test[reference_index], rtol=1e-5, atol=1e-6)
    ref_scores = probabilities(exported.run(["logits"], {"feature": ref_vector[None]})[0])[0]
    np.testing.assert_allclose(ref_scores, exported_scores[reference_index], rtol=1e-5, atol=1e-6)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    head_path = OUTPUT_DIR / "BloomFace_LibreFace_Head.onnx"
    onnx.save(model, head_path)
    settings = {
        "classes": list(CLASSES),
        "preprocess": PREPROCESS,
        "artifact_sha256": {name: spec[1] for name, spec in ARTIFACTS.items()},
        "head_sha256": sha256(head_path),
        "feature_length": features,
        "policy": policy,
        "reference": {
            "image_sha256": entry[4],
            "target": CLASSES[entry[3]],
            "prediction": CLASSES[exported_ids[reference_index]] if exported_ids[reference_index] != -1 else None,
            "scores": ref_scores.tolist(),
        },
    }
    (OUTPUT_DIR / "BloomFace_LibreFace_Config.json").write_text(json.dumps(settings, indent=2) + "\n", encoding="utf-8")
    print(f"Paridad confirmada en {len(x_test)} imágenes. Diferencia máxima de puntajes: {np.max(np.abs(exported_scores - test_scores)):.2e}.")
    print(f"Referencia: etiqueta {settings['reference']['target']}, predicción {settings['reference']['prediction']}, puntajes {np.round(ref_scores, 6).tolist()}.")
    print("Exportados: BloomFace_LibreFace_Head.onnx y BloomFace_LibreFace_Config.json.")
else:
    print("Exportación omitida: falta una partición viable de entrenamiento, validación y prueba.")
analyzer.close()
del analyzer

Paridad confirmada en 300 imágenes. Diferencia máxima de puntajes: 1.79e-07.
Referencia: etiqueta boca_abierta, predicción boca_abierta, puntajes [0.00031599999056197703, 0.00015700000221841037, 0.9982140064239502, 0.0013139999937266111].
Exportados: BloomFace_LibreFace_Head.onnx y BloomFace_LibreFace_Config.json.
